# 04 — Un réseau de neurones à l'œuvre : l'autoencodeur

**IA-01 · Jour 4 · Hexalp**

Hugo a entraîné un **autoencodeur** : un petit réseau de neurones qui a appris à **recopier** les connexions normales de septembre, en les faisant passer par un goulot de 3 neurones. Ce qu'il a l'habitude de voir, il le recopie bien. Ce qui est inhabituel, il le recopie mal : son **erreur de reconstruction** devient un score d'anomalie.

Vous n'entraînez rien : vous chargez le modèle de Hugo et vous l'utilisez. C'est ce que vous ferez le plus souvent avec un modèle de Deep Learning.

```
 7 caractéristiques → 8 neurones → 3 neurones → 8 neurones → 7 caractéristiques reconstruites
     (entrée)                       (goulot)                          (sortie)
```

---
## 1. Vérifier avant de charger

Un modèle Scikit-learn s'enregistre avec **joblib**, qui s'appuie sur le format **pickle** de Python. Charger un tel fichier, c'est **exécuter du code** : un fichier piégé peut faire n'importe quoi sur votre machine. On vérifie donc son empreinte SHA-256 avant de l'ouvrir. L'empreinte attendue vous a été transmise par un autre canal que le fichier : elle figure dans le kit, séquence 4.7.

In [ ]:
import hashlib

EMPREINTE_ATTENDUE = "___"

with open("autoencodeur_hexalp.joblib", "rb") as f:
    empreinte = hashlib.sha256(f.read()).hexdigest()

print("Empreinte du fichier :", empreinte)
if empreinte != EMPREINTE_ATTENDUE:
    raise ValueError("Empreinte différente : ne chargez pas ce fichier, prévenez le SOC.")
print("✅ Empreinte conforme")

In [ ]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

modele = joblib.load("autoencodeur_hexalp.joblib")
print(modele["description"])

normaliseur = modele["normaliseur"]
reseau = modele["reseau"]
SEUIL = modele["seuil"]
CARACTERISTIQUES = modele["caracteristiques"]

---
## 2. Ouvrir la boîte

Le réseau est fait de couches de neurones. Chaque connexion entre deux neurones porte un **poids**, et chaque neurone un **biais** : ce sont les paramètres trouvés pendant l'entraînement.

In [ ]:
print("Neurones par couche :", [reseau.n_features_in_] + list(reseau.hidden_layer_sizes) + [reseau.n_outputs_])

nb_parametres = sum(poids.size for poids in reseau.coefs_) + sum(biais.size for biais in reseau.intercepts_)
print("Nombre de paramètres :", ___)

In [ ]:
# Les 24 poids entre l'entrée et la première couche cachée
pd.DataFrame(reseau.coefs_[0], index=CARACTERISTIQUES, columns=[f"neurone {i+1}" for i in range(8)]).round(2)

✍️ **R1.** Combien de paramètres compte ce réseau ? Combien de seuils comptait votre arbre de décision ? Regardez le tableau des poids : pouvez-vous dire, comme avec l'arbre, « le réseau alerte quand… » ?

---
## 3. Utiliser le modèle sur octobre

In [ ]:
octobre = pd.read_csv("hexalp_ssh_oct2026.csv")
octobre["octets_sortants_mo"] = octobre["octets_sortants_mo"].fillna(octobre["octets_sortants_ko"] / 1024)
octobre["octets_sortants_mo"] = octobre["octets_sortants_mo"].fillna(octobre["octets_sortants_mo"].median())
octobre["externe"] = (octobre["origine"] == "externe").astype(int)

X = octobre[CARACTERISTIQUES]
Z = normaliseur.transform(X)                      # mise à l'échelle, comme pendant l'entraînement
reconstruction = reseau.predict(Z)                # le réseau tente de recopier chaque ligne
erreurs = (Z - reconstruction) ** 2               # l'écart, caractéristique par caractéristique

octobre["erreur"] = erreurs.mean(axis=1)
octobre["anomalie_ae"] = (octobre["erreur"] > ___).astype(int)

pd.crosstab(octobre["anomalie_ae"], octobre["etiquette"])

In [ ]:
# Pour comparer : l'Isolation Forest du J3, entraîné sur septembre
septembre = pd.read_csv("hexalp_ssh_sept2026.csv")
septembre["octets_sortants_mo"] = septembre["octets_sortants_mo"].fillna(septembre["octets_sortants_mo"].median())
septembre["externe"] = (septembre["origine"] == "externe").astype(int)
iforest = IsolationForest(contamination=0.05, random_state=42).fit(septembre[CARACTERISTIQUES])
octobre["anomalie_if"] = (iforest.predict(X) == -1).astype(int)

groupes = {
    "Sauvegarde nocturne (normal)": octobre["ip_source"] == "10.20.8.30",
    "Proxy Zabbix (normal)": octobre["ip_source"] == "10.20.1.7",
    "Exfiltrations par VPN (attaque)": (octobre["etiquette"] == "attaque") & octobre["ip_source"].str.startswith("10.20.250."),
    "Brute force (attaque)": (octobre["etiquette"] == "attaque") & (octobre["echecs_auth"] >= 40) & (octobre["succes_auth"] == 0),
    "Toutes les attaques": octobre["etiquette"] == "attaque",
    "Toutes les lignes normales": octobre["etiquette"] == "normal",
}
pd.DataFrame({nom: {"lignes": masque.sum(),
                    "signalées par l'Isolation Forest": octobre.loc[masque, "anomalie_if"].sum(),
                    "signalées par l'autoencodeur": octobre.loc[masque, "anomalie_ae"].sum()}
              for nom, masque in groupes.items()}).T

✍️ **R2.** Où l'autoencodeur fait-il mieux que l'Isolation Forest ? Où fait-il moins bien ? Rappelez-vous sur quelles données chacun a été entraîné.

---
## 4. Expliquer une alerte

Le réseau ne donne pas de règle. Mais on peut regarder **quelle caractéristique il a le plus de mal à recopier**.

In [ ]:
def detail(ligne):
    i = octobre.index.get_loc(ligne)
    return pd.DataFrame({"valeur": X.iloc[i], "erreur de reconstruction": erreurs[i].round(2)})

# L'exfiltration du 9 octobre à 3 h, repérée à la main par Hugo
exfiltration = octobre[(octobre["date"] == "2026-10-09") & (octobre["heure"] == 3) & octobre["ip_source"].str.startswith("10.20.250.")].index[0]
detail(exfiltration)

In [ ]:
# Une nuit de sauvegarde ordinaire
sauvegarde = octobre[octobre["ip_source"] == "10.20.8.30"].index[___]
detail(sauvegarde)

✍️ **R3.** Qu'est-ce qui « dérange » le réseau dans l'exfiltration du 9 octobre ? Cette explication suffirait-elle à Nadia pour décider de couper un accès VPN ?

✍️ **R4.** Ce fichier a été entraîné par Hugo, sur un poste du SOC, puis copié sur votre machine. Citez trois façons dont il aurait pu être modifié en chemin. Que vérifie l'empreinte SHA-256 ? Que ne vérifie-t-elle pas ?

✍️ **R5.** Les grands modèles que vous utilisez tous les jours (J1) sont des réseaux du même type, avec des milliards de paramètres au lieu de 186. Que retenez-vous de cette séquence pour expliquer leurs décisions ou vérifier les fichiers que vous téléchargez ?

---
✅ **Fin du notebook.**